# Generate Projections and Blank Labels
Isobel Taylor-Hearn, January 2026

In [3]:
import contextlib, joblib, time
from joblib import Parallel, delayed
from tqdm import tqdm

import numpy as np
import pandas as pd
import tifffile as tiff
import tifffile
from pathlib import Path
from skimage.io import imread
from skimage.transform import downscale_local_mean
from skimage.filters import gaussian, laplace
from liffile import LifFile


In [20]:
# Update the `process_file` function to log detailed error messages

def process_file(path_str: str, downscale: int, out_root_str: str):
    p = Path(path_str)
    out_root = Path(out_root_str)
    rows = []

    def run_one(base, z_stack, x_um, y_um):
        out1 = out_root / "in_focus_full" / f"{base}_PROJECTION.tif"
        out2 = out_root / "in_focus_masked" / f"{base}_PROJECTION.tif"
        out3 = out_root / "in_focus_each_point" / f"{base}_PROJECTION.tif"
        lbl  = out_root / "labels" / f"{base}_LABELS.tif"

        # skip everything if already done
        if out1.exists() and out2.exists() and out3.exists() and lbl.exists():
            rows.append({
                "file": base, "method": "SKIPPED",
                "z_min": None, "z_max": None,
                "n_z_slices_used": None,
                "time_sec": 0.0,
                "Z": None, "downscale": downscale
            })
            return

        try:
            # ---- coerce to (Z, Y, X) ----
            z_stack = np.asarray(z_stack)
            while z_stack.ndim > 3:
                z_stack = z_stack[0]
            if z_stack.ndim == 2:
                z_stack = z_stack[None, ...]

            # ---- downscale XY only ----
            z_ds = downscale_local_mean(z_stack, (1, downscale, downscale))
            x_um_ds = (x_um * downscale) if x_um is not None else None
            y_um_ds = (y_um * downscale) if y_um is not None else None

            mask = center_exclusion_mask(z_ds.shape[1:], center_frac=0.25)

            # ---- option 1: global best-focus ----
            t0 = time.perf_counter()
            z1, proj1 = best_focus_slice(z_ds, mask=None)
            dt = time.perf_counter() - t0
            save_tif(proj1, x_um_ds, y_um_ds, out1)
            rows.append({
                "file": base, "method": "in_focus_full",
                "z_min": int(z1), "z_max": int(z1),
                "n_z_slices_used": 1,
                "time_sec": dt,
                "Z": int(z_ds.shape[0]),
                "downscale": downscale
            })

            # ---- option 2: masked best-focus ----
            t0 = time.perf_counter()
            z2, proj2 = best_focus_slice(z_ds, mask=mask)
            dt = time.perf_counter() - t0
            save_tif(proj2, x_um_ds, y_um_ds, out2)
            rows.append({
                "file": base, "method": "in_focus_masked",
                "z_min": int(z2), "z_max": int(z2),
                "n_z_slices_used": 1,
                "time_sec": dt,
                "Z": int(z_ds.shape[0]),
                "downscale": downscale
            })

            # ---- option 3: tilted plane refocus ----
            t0 = time.perf_counter()
            proj3, zmap = tilted_plane_refocus(z_ds, grid=10, patch=32, mask=mask)
            dt = time.perf_counter() - t0
            save_tif(proj3, x_um_ds, y_um_ds, out3)
            rows.append({
                "file": base, "method": "in_focus_each_point",
                "z_min": int(zmap.min()),
                "z_max": int(zmap.max()),
                "n_z_slices_used": int(np.unique(zmap).size),
                "time_sec": dt,
                "Z": int(z_ds.shape[0]),
                "downscale": downscale
            })

            # ---- single label only ----
            save_label_once(proj1.shape, x_um_ds, y_um_ds, lbl)

        except Exception as e:
            rows.append({
                "file": base, "method": "ERROR_PROCESS",
                "z_min": None, "z_max": None,
                "n_z_slices_used": None,
                "time_sec": None,
                "Z": None, "downscale": downscale,
                "error": repr(e)
            })

    # -------------------------
    # TIFF input
    # -------------------------
    if p.suffix.lower() in (".tif", ".tiff"):
        try:
            run_one(p.stem, imread(p), *get_tif_pixel_size(p))
        except Exception as e:
            rows.append({
                "file": p.stem, "method": "ERROR_READ",
                "z_min": None, "z_max": None,
                "n_z_slices_used": None,
                "time_sec": None,
                "Z": None, "downscale": downscale,
                "error": repr(e)
            })

    # -------------------------
    # LIF input
    # -------------------------
    elif p.suffix.lower() == ".lif":
        try:
            with LifFile(p) as lif:
                for i, im in enumerate(lif.images):
                    base = f"{p.stem}_img{i}"
                    try:
                        z = im.asarray()
                        if im.dims == ("C", "Z", "Y", "X"):
                            z = z[0]
                        run_one(base, z, *get_lif_pixel_size(im))
                    except Exception as e:
                        rows.append({
                            "file": base, "method": "ERROR_READ",
                            "z_min": None, "z_max": None,
                            "n_z_slices_used": None,
                            "time_sec": None,
                            "Z": None, "downscale": downscale,
                            "error": repr(e)
                        })
                        continue
        except Exception as e:
            rows.append({
                "file": p.stem, "method": "ERROR_OPEN",
                "z_min": None, "z_max": None,
                "n_z_slices_used": None,
                "time_sec": None,
                "Z": None, "downscale": downscale,
                "error": repr(e)
            })

    return rows

In [21]:
# -------------------------
# run (guard for Windows)
# -------------------------
if __name__ == "__main__":
    files = [str(p) for p in sorted(in_dir.iterdir())
                if p.suffix.lower() in (".tif", ".tiff", ".lif")]
    N_JOBS = 6

    try:
        with tqdm_joblib(tqdm(desc="Processing files", total=len(files))) as _:
            all_rows = Parallel(n_jobs=N_JOBS, backend="loky")(
                delayed(process_file)(fp, DOWNSCALE_FACTOR, str(out_root))
                for fp in files
            )

    except Exception as e:
        # catastrophic failure (joblib / worker startup)
        print("Fatal error during parallel execution:")
        print(repr(e))
        raise

    # flatten rows, capturing worker-level failures cleanly
    rows = []
    for fp, result in zip(files, all_rows):
        try:
            rows.extend(result)
        except Exception as e:
            rows.append({
                "file": Path(fp).stem,
                "method": "ERROR",
                "z_min": None,
                "z_max": None,
                "n_z_slices_used": None,
                "time_sec": None,
                "Z": None,
                "downscale": DOWNSCALE_FACTOR,
                "error": repr(e),
            })
    df = pd.DataFrame(rows)

    # Save the output to a CSV file
    output_file = "output_rows.csv"  # Specify the desired file name
    df.to_csv(output_file, index=False)
    print(f"Output saved to {output_file}")


Processing files:   0%|          | 0/93 [00:00<?, ?it/s]

Processing files: 100%|██████████| 93/93 [1:21:45<00:00, 52.74s/it] 

Output saved to output_rows.csv


In [22]:
import pandas as pd

# Convert rows to a DataFrame
df = pd.DataFrame(rows)

# Save the output to a CSV file
output_file = "output_rows.csv"  # Specify the desired file name
df.to_csv(output_file, index=False)
print(f"Output saved to {output_file}")

Output saved to output_rows.csv


In [23]:
df

,file,method,z_min,z_max,n_z_slices_used,time_sec,Z,downscale,error
0,20250619_Vascumap_Dev25_11_Daisy10,SKIPPED,None,None,None,0.0,None,4,NaN
1,20250619_Vascumap_Dev25_11_Tulip10,SKIPPED,None,None,None,0.0,None,4,NaN
2,20250619_Vascumap_Dev25_11_Wicell10,SKIPPED,None,None,None,0.0,None,4,NaN
3,20250619_Vascumap_Dev25_11_Wicell10_2,SKIPPED,None,None,None,0.0,None,4,NaN
4,20250619_Vascumap_Dev25_11_WicellRescue,SKIPPED,None,None,None,0.0,None,4,NaN
...,...,...,...,...,...,...,...,...,...
436,marina_M7_2025.12.05_FL39_img19,SKIPPED,None,None,None,0.0,None,4,NaN
437,marina_M7_2025.12.05_FL39_img20,ERROR_PROCESS,None,None,None,NaN,None,4,"PermissionError(13, 'Permission denied')"
438,marina_M7_2025.12.05_FL39_img21,ERROR_PROCESS,None,None,None,NaN,None,4,"PermissionError(13, 'Permission denied')"
439,marina_M7_2025.12.05_FL39_img22,ERROR_PROCESS,None,None,None,NaN,None,4,"PermissionError(13, 'Permission denied')"


In [14]:
df["method"].unique()

array(['ERROR_READ'], dtype=object)

In [ ]:




# for file in sorted(in_dir.iterdir()):
#     print(file)
#     if file.suffix.lower() in ('.tif', '.tiff'):
#         x_pixel_size_um, y_pixel_size_um = get_tif_pixel_size(file)
#         z_stack = imread(file)
#     elif file.suffix.lower() == ".lif":
#         with LifFile(file) as lif:
#             for i, image in enumerate(lif.images):
#                 # handle channel if present
#                 x_pixel_size_um, y_pixel_size_um = get_lif_pixel_size(image)
#                 z_stack = image.asarray()
#                 if image.dims == ("C", "Z", "Y", "X"):
#                     z_stack = z_stack[0]  # (Z, Y, X)
#     else:
#         continue
#     print(f"Image shape {z_stack.shape}, pixel size {x_pixel_size_um} x {y_pixel_size_um}")
#     downscaled_z_stack = downscale_local_mean(z_stack, (1, DOWNSCALE_FACTOR, DOWNSCALE_FACTOR))
    
#     blank_labels = np.zeros(downscaled_z_stack.shape[0,:,:], dtype=np.uint8)
    
#         out_lab_path = lab_out / f"{p.stem}_labels.tif"
#         tiff.imwrite(out_lab_path, lab)
    
    
    




In [ ]:

                    
                    
                    
    # with LifFile(lif_path) as lif:
    #     number_of_lifs = len(lif.images)

    #     for i in range(number_of_lifs):
    #         print("{}, {}".format(i, lif_name))   
    #         try:      
    #             img = lif.images[i]
    #             print(img.dims)
    #             if img.dims == ('C', 'Z', 'Y', 'X'):
    #                 print("image shape is {}".format(img.shape))
    #                 image = img.asarray()
    #                 image = image[0, :, :, :]  # Take first channel
    #                 print("image shape is {}".format(image.shape))

    #                 max_proj = np.max(image, axis=0)
    #                 xa = img.asxarray()

    #                 # liffile coordinates are typically in meters → convert to µm
    #                 x_um = step("X", xa) * 1e6 if step("X", xa) is not None else None
    #                 print(f"step size um {x_um}")
    #                 out_path = os.path.join(
    #                     output_dir,
    #                     f"{lif_name}_img{i}_maxproj.ome.tif"
    #                 )

    #                 tiff.imwrite(
    #                     out_path,
    #                     max_proj.astype(np.float32),
    #                     ome=True,
    #                     metadata={
    #                         "PhysicalSizeX": x_um,
    #                         "PhysicalSizeY": x_um,
    #                         "PhysicalSizeXUnit": "µm",
    #                         "PhysicalSizeYUnit": "µm",
    #                     }
    #                 )
    #             if img.dims == ('Z', 'Y', 'X'):
    #                 image = img.asarray()

    #                 max_proj = np.max(image, axis=0)
    #                 xa = img.asxarray()

    #                 # liffile coordinates are typically in meters → convert to µm
    #                 x_um = step("X", xa) * 1e6 if step("X", xa) is not None else None
    

    #         max_proj = np.max(img, axis=0)  # Z projection
    #         z_proj = np.max(img, axis=1)  # Y projection

    #         out_path_max = os.path.join(
    #             output_dir,
    #             f"{tif_name}_img{i}_maxproj.ome.tif"
    #         )
    #         out_path_z = os.path.join(
    #             output_dir,
    #             f"{tif_name}_img{i}_zproj.ome.tif"
    #         )

    #         tiff.imwrite(
    #             out_path_max,
    #             max_proj.astype(np.float32),
    #             ome=True,
    #             metadata={
    #                 "PhysicalSizeX": x_um,
    #                 "PhysicalSizeY": y_um,
    #                 "PhysicalSizeXUnit": "µm",
    #                 "PhysicalSizeYUnit": "µm",
    #             }
    #         )

    #     except Exception as e:
    #         print(f"Could not process image {i} in {tif_name}: {e}")

    # elif file.suffix.lower() == '.lif':



    # else:
    #     continue
    
    

    # def _tiff_series(path: Path):
    #     name = path.stem.lower().replace('.', '_')
    #     try:
    #         arr = tiff.imread(path)
    #     except Exception:
    #         return []
    #     # optional pixel size
    #     x_um = y_um = None
    #     try:
    #         with tiff.TiffFile(path) as tif:
    #             tags = tif.pages[0].tags
    #             if 'XResolution' in tags and 'YResolution' in tags:
    #                 xres = tags['XResolution'].value
    #                 yres = tags['YResolution'].value
    #                 x_um = 1 / xres[0] * xres[1] * 1e6
    #                 y_um = 1 / yres[0] * yres[1] * 1e6
    #     except Exception:
    #         pass

    #     if arr.ndim == 4 and arr.shape[0] <= 8:  # likely C,Z,Y,X
    #         arr = arr[0]
    #     if arr.ndim in (2, 3):
    #         return [(name, 0, arr, x_um, y_um)]
    #     return []

    # def _lif_series(path: Path):
    #     name = path.stem.lower().replace('.', '_')
    #     outs = []
    #     with LifFile(path) as lif:
    #         for i, img in enumerate(lif.images):
    #             try:
    #                 arr = img.asarray()
    #             except Exception:
    #                 continue
    #             # drop channel if present
    #             if img.dims == ('C', 'Z', 'Y', 'X') and arr.ndim == 4:
    #                 arr = arr[0]
    #             xa = None
    #             try:
    #                 xa = img.asxarray()
    #             except Exception:
    #                 xa = None
    #             x_um = step('X', xa) * 1e6 if xa is not None and step('X', xa) is not None else None
    #             if arr.ndim in (2, 3):
    #                 outs.append((name, i, arr, x_um, x_um))
    #     return outs

    # def _save(path: Path, arr: np.ndarray, x_um=None, y_um=None):
    #     meta = {
    #         "PhysicalSizeX": x_um,
    #         "PhysicalSizeY": y_um,
    #         "PhysicalSizeXUnit": "µm",
    #         "PhysicalSizeYUnit": "µm",
    #     }
    #     tiff.imwrite(str(path), arr.astype(np.float32), ome=True, metadata=meta)

    # for p in sorted(folder.iterdir()):
    #     if p.suffix.lower() in ('.tif', '.tiff'):
    #         series = _tiff_series(p)
    #     elif p.suffix.lower() == '.lif':
    #         series = _lif_series(p)
    #     else:
    #         continue

    #     for name, idx, arr, x_um, y_um in series:
    #         if arr.ndim == 3:
    #             max_proj = np.max(arr, axis=0)
    #         else:
    #             max_proj = arr

    #         out_name = f"{name}_img{idx}_maxproj.ome.tif" if p.suffix.lower() == '.lif' else f"{name}_maxproj.ome.tif"
    #         out_path = out_dir / out_name

    #         if out_path.exists() and not overwrite:
    #             if verbose:
    #                 print(f"Skipping existing: {out_path}")
    #             continue

    #         try:
    #             _save(out_path, max_proj, x_um, y_um)
    #             if verbose:
    #                 print(f"Saved: {out_path}")
    #         except Exception as e:
    #             print(f"Could not save {out_path}: {e}")


# Example usage:
# process_images(r"Z:\\Bel\\Vascumap_Example_Lifs\\done\\marina",
#                r"Z:\\Bel\\Vascumap_Example_Lifs\\max_projections\\unseen_images",
#                overwrite=False,
#                verbose=True)
